# SereBench — Systematic W&B Experiment Plots

Fetches (almost) **all** scalar metrics logged to the W&B project
`jfcevallos/SereBench` by `python tests/experiments_all.py`, groups the runs by
**experiment configuration** (W&B `group`) and **seed**, and plots
**mean ± 95 % confidence interval across the 5 seeds** for every metric family,
every vehicle, every classifier architecture and every FL strategy.

This notebook is the SereBench analogue of `lion_plots.py`: the same
statistically-grounded *t*-distribution confidence bands, the same bold
high-contrast typography, but driven by the SereBench experiment matrix
instead of the LION agents.

**Requires** a `.env` file in the repo root containing:
```
WANDB_API_KEY=...
# optional, defaults to jfcevallos / SereBench
WANDB_ENTITY=jfcevallos
WANDB_PROJECT_NAME=SereBench
```
Install deps: `pip install wandb pandas numpy scipy matplotlib python-dotenv`

### Experiment matrix (W&B groups)
`experiments_all.py` produces **10 base experiments × 3 architectures = 30 groups**
(× 5 seeds = up to 150 runs). The `mlp` architecture has **no** group suffix;
`cnn`/`resnet` append `-cnn` / `-resnet`. Run names are
`{group}_seed{seed}_run{idx}`.

| # | group | FL | FL strategy | Adv. training |
|---|-------|----|-------------|---------------|
| 1 | `noadvtraining-nofl`    | – | – | – |
| 2 | `advtraining-nofl`      | – | – | ✓ |
| 3 | `noadvtraining-fl`      | ✓ | FedAvg  | – |
| 4 | `noadvtraining-fedprox` | ✓ | FedProx | – |
| 5 | `noadvtraining-fedyogi` | ✓ | FedYogi | – |
| 6 | `advtraining-fl`        | ✓ | FedAvg  | ✓ (not Angela) |
| 7 | `advtraining-fedprox`   | ✓ | FedProx | ✓ (not Angela) |
| 8 | `advtraining-fedyogi`   | ✓ | FedYogi | ✓ (not Angela) |
| 9 | `dynamic-noise-fl`      | ✓ | FedAvg  | ✓ (Angela noise 0→1 mid-run) |
| 10| `dynamic-noise-nofl`    | – | – | ✓ (Angela noise 0→1 mid-run) |

### What gets plotted
- **In-distribution training** (`class_*`): loss, accuracy, precision, recall, F1, macro-F1
- **Online monitoring** (`online_class_*`) — ET4
- **Adversarial evaluation — Gaussian noise** (`adv_eval_*`) — ET1, ET2, ET3
- **Adversarial evaluation — HSJA** (`hsja_adv_eval/*`): metrics, avg L2 perturbation, avg queries
- **Attack-mitigation** (`mitigation_reward`, `mitigation_time`)
- Comparisons across: **adv-training on/off**, **FL on/off**, **FedAvg vs FedProx vs FedYogi**,
  **MLP vs CNN vs ResNet**, the **four canonical configurations**, and the **dynamic-noise** runs.

> Plotly artifacts (`*_manifold_*`, `*_confusion_matrix`) are logged as W&B *media*,
> not scalar history; they are best inspected in the W&B UI and are intentionally
> not re-fetched here. The last section lists which runs carry them.


## 1 — Imports & typography

In [ ]:
pip install --upgrade wandb

In [ ]:
import os
import re
import itertools
import warnings
import pickle
from pathlib import Path
from math import ceil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from dotenv import load_dotenv
from scipy import stats as scipy_stats
import wandb

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Bold, large typography (mirrors lion_plots.py)
title_font  = {'weight': 'bold', 'size': 22}
axis_font   = {'weight': 'bold', 'size': 18}
legend_font = {'weight': 'bold', 'size': 16}
plt.rcParams['axes.linewidth'] = 1.6
plt.rcParams['figure.facecolor'] = 'white'
print('Imports OK')

## 2 — Experiment / metric catalogue

Encodes the fleet (and its canonical noise gradient), the architectures, the 10
base experiment groups and the full set of metric suffixes logged under
`{vehicle}_statistics.*` (W&B flattens nested dicts with `.`; HSJA keys keep the
internal `/` separator).

In [ ]:
# Fleet and canonical adversarial-noise std (Table 1)
VEHICLES = ['angela', 'bob', 'claude', 'daniel']
VEHICLE_NOISE = {'angela': 0.0, 'bob': 0.7, 'claude': 1.4, 'daniel': 2.1}

# Classifier architectures (mlp = no group suffix; cnn/resnet add a suffix)
MODELS = ['mlp', 'cnn', 'resnet']

# The 10 base experiment groups (mlp variant; add -cnn/-resnet for the others)
BASE_EXPERIMENTS = [
    'noadvtraining-nofl', 'advtraining-nofl',
    'noadvtraining-fl', 'noadvtraining-fedprox', 'noadvtraining-fedyogi',
    'advtraining-fl', 'advtraining-fedprox', 'advtraining-fedyogi',
    'dynamic-noise-fl', 'dynamic-noise-nofl',
]

DEFAULT_SEEDS = [42, 123, 456, 789, 1234]

# Metric suffixes (appended after f"{vehicle}_statistics.")
TRAINING_METRICS   = ['total_loss', 'class_accuracy', 'class_precision',
                      'class_recall', 'class_f1', 'class_macro_f1']
ONLINE_METRICS     = ['online_class_accuracy', 'online_class_precision',
                      'online_class_recall', 'online_class_f1', 'online_class_macro_f1']
ADV_GAUSS_METRICS  = ['adv_eval_accuracy', 'adv_eval_precision', 'adv_eval_recall',
                      'adv_eval_f1', 'adv_eval_macro_f1']
HSJA_METRICS       = ['hsja_adv_eval/accuracy', 'hsja_adv_eval/precision',
                      'hsja_adv_eval/recall', 'hsja_adv_eval/f1', 'hsja_adv_eval/macro_f1',
                      'hsja_adv_eval/avg_perturbation', 'hsja_adv_eval/avg_queries']
MITIGATION_METRICS = ['mitigation_reward', 'mitigation_time']
COUNTER_METRICS    = ['records_processed', 'anoms_processed',
                      'attacks_processed', 'diagnostics_processed']

def metric_key(vehicle, suffix):
    # Full W&B history column for a vehicle metric.
    return f"{vehicle}_statistics.{suffix}"

_PRETTY = {
    'total_loss': 'Training Loss',
    'class_accuracy': 'Accuracy (train)', 'class_precision': 'Precision (train)',
    'class_recall': 'Recall (train)', 'class_f1': 'F1 weighted (train)',
    'class_macro_f1': 'Macro-F1 (train)',
    'online_class_accuracy': 'Online Accuracy', 'online_class_precision': 'Online Precision',
    'online_class_recall': 'Online Recall', 'online_class_f1': 'Online F1 (weighted)',
    'online_class_macro_f1': 'Online Macro-F1',
    'adv_eval_accuracy': 'Adv-eval Accuracy', 'adv_eval_precision': 'Adv-eval Precision',
    'adv_eval_recall': 'Adv-eval Recall', 'adv_eval_f1': 'Adv-eval F1 (weighted)',
    'adv_eval_macro_f1': 'Adv-eval Macro-F1',
    'hsja_adv_eval/accuracy': 'HSJA Accuracy', 'hsja_adv_eval/precision': 'HSJA Precision',
    'hsja_adv_eval/recall': 'HSJA Recall', 'hsja_adv_eval/f1': 'HSJA F1 (weighted)',
    'hsja_adv_eval/macro_f1': 'HSJA Macro-F1',
    'hsja_adv_eval/avg_perturbation': 'HSJA avg L2 perturbation',
    'hsja_adv_eval/avg_queries': 'HSJA avg queries',
    'mitigation_reward': 'Mitigation Reward', 'mitigation_time': 'Mitigation Time (s)',
    'records_processed': 'Records processed', 'anoms_processed': 'Anomalies processed',
    'attacks_processed': 'Attacks processed', 'diagnostics_processed': 'Normals processed',
}

def pretty(suffix):
    return _PRETTY.get(suffix, suffix.replace('_', ' ').title())

def ylim_for(suffix):
    s = suffix.lower()
    if any(t in s for t in ['accuracy', 'precision', 'recall', 'f1']):
        return (0, 1.02)
    if any(t in s for t in ['loss', 'perturbation', 'queries', 'time', 'processed']):
        return (0, None)
    return None

print('Catalogue ready:', len(BASE_EXPERIMENTS), 'base experiments,',
      len(MODELS), 'architectures,', len(VEHICLES), 'vehicles')

## 3 — W&B API & history fetch

Fetches every run in the project, keyed by `(group, seed)`. The W&B `group` is
set natively at `wandb.init(group=...)`; the seed is parsed from the run name
(`..._seed<N>_run<M>`), falling back to the run config.

Results are cached to `_serebench_history_cache.pkl` so re-running the plotting
cells is instant. Set `FORCE_REFETCH = True` (or delete the cache file) to pull
fresh data, and `GROUPS_FILTER` to restrict the fetch to a subset of groups.

In [ ]:
# Locate and load the .env (searches upward from the notebook's directory)
load_dotenv()
for parent in [Path.cwd()] + list(Path.cwd().parents):
    cand = parent / '.env'
    if cand.exists():
        load_dotenv(cand, override=False)
        break

WANDB_ENTITY       = os.getenv('WANDB_ENTITY', 'jfcevallos')
WANDB_PROJECT_NAME = os.getenv('WANDB_PROJECT_NAME', 'SereBench')
WANDB_PROJECT      = f"{WANDB_ENTITY}/{WANDB_PROJECT_NAME}"

HISTORY_SAMPLES = 100_000      # per-run history cap (raise if runs are very long)
CACHE_PATH      = Path('_serebench_history_cache.pkl')
FORCE_REFETCH   = False
GROUPS_FILTER   = None         # e.g. ['noadvtraining-nofl'] ; None = all groups
WANDB_TAG_FILTER = ['BROKEN_HSJA']       # e.g. ['my_tag'] ; None = all tags

def _seed_of(run):
    m = re.search(r'_seed(\d+)', run.name or '')
    if m:
        return int(m.group(1))
    cfg = run.config or {}
    for path in (('default_consumer_config', 'seed'), ('default_vehicle_config', 'seed')):
        d = cfg
        try:
            for p in path:
                d = d[p]
            return int(d)
        except Exception:
            pass
    return -1

def fetch_runs():
    api = wandb.Api(timeout=60)
    runs = api.runs(WANDB_PROJECT)
    out = {}
    print(f'Scanning runs in {WANDB_PROJECT} ...')
    for run in runs:
        grp = run.group
        if grp is None:
            continue
        if GROUPS_FILTER and grp not in GROUPS_FILTER:
            continue
        if WANDB_TAG_FILTER and not any(tag in run.tags for tag in WANDB_TAG_FILTER):
            continue
        seed = _seed_of(run)
        try:
            hist = run.history(samples=HISTORY_SAMPLES)
        except Exception as exc:
            print(f'  ! history failed for {run.name}: {exc}')
            continue
        if '_step' in hist.columns:
            hist = hist.set_index('_step')
        # If (group, seed) already seen, keep the longer history (most complete run)
        prev = out.get((grp, seed))
        if prev is None or len(hist) >= len(prev):
            out[(grp, seed)] = hist
        print(f'  {grp:30s} seed={seed:<5} steps={len(hist):>6}  ({run.id})')
    return out

if (not FORCE_REFETCH) and CACHE_PATH.exists():
    print('Loading cached histories from', CACHE_PATH)
    with open(CACHE_PATH, 'rb') as fh:
        RUNS = pickle.load(fh)
else:
    RUNS = fetch_runs()
    with open(CACHE_PATH, 'wb') as fh:
        pickle.dump(RUNS, fh)
    print('Cached to', CACHE_PATH)

print(f'\nTotal (group, seed) run-histories loaded: {len(RUNS)}')


## 4 — Discovery: which groups & metrics are present?

Lists the groups actually found,
the seeds per group, and every distinct metric column — so ONE can sanity-check
coverage before plotting and discover any keys not hard-listed above.

In [ ]:
groups_present = sorted({g for (g, s) in RUNS})
GROUPS_PRESENT = set(groups_present)

print(f'Groups present ({len(groups_present)}):')
for g in groups_present:
    seeds = sorted({s for (gg, s) in RUNS if gg == g})
    print(f'  {g:30s} seeds={seeds}  (n={len(seeds)})')

all_cols = sorted({c for h in RUNS.values() for c in h.columns})
print(f'\nTotal distinct metric columns across all runs: {len(all_cols)}')

# Show the per-vehicle statistics columns (what we actually plot)
for v in VEHICLES:
    vc = [c for c in all_cols if c.startswith(v + '_statistics')]
    if not vc:
        continue
    print(f'\n--- {v}: {len(vc)} statistics columns ---')
    for c in vc:
        print('   ', c)

# Any non-vehicle-statistics scalar columns (e.g. global_metrics.*) for awareness
other = [c for c in all_cols
         if not any(c.startswith(v + '_statistics') for v in VEHICLES)
         and not c.startswith('_')]
if other:
    print(f'\n--- other / global columns ({len(other)}) ---')
    for c in other:
        print('   ', c)

## 5 — Plotting engine (CI-aware)

`seed_series_for_group` collects one column per seed for a given metric (drops
NaNs — vehicles share a global W&B step axis — and re-indexes to a per-vehicle
reporting ordinal). `compute_ci_band` returns the mean and a **95 % *t*-CI across
seeds**. `_plot_grid` lays out a grid of panels, each panel overlaying several
series (mean line + shaded CI) with a shared legend.

In [ ]:
def seed_series_for_group(group, key, smooth=1):
    # Returns a wide DataFrame: index = per-vehicle reporting ordinal, columns = seeds.
    cols = {}
    for (g, seed), hist in RUNS.items():
        if g != group or key not in hist.columns:
            continue
        s = pd.to_numeric(hist[key], errors='coerce').dropna().reset_index(drop=True)
        if smooth > 1:
            s = s.rolling(smooth, min_periods=1).mean()
        if len(s):
            cols[f'seed{seed}'] = s
    if not cols:
        return None
    return pd.DataFrame(cols)

def compute_ci_band(wide, alpha=0.95):
    # mean ± t_{n-1,(1+alpha)/2} * std / sqrt(n), computed row-wise across seeds.
    if wide is None or wide.shape[1] == 0:
        return None, None, None
    w = wide.apply(pd.to_numeric, errors='coerce')
    mean = w.mean(axis=1)
    std  = w.std(axis=1, ddof=1)
    n    = w.notna().sum(axis=1)
    t_mult = pd.Series(
        [scipy_stats.t.ppf((1 + alpha) / 2, df=max(int(ni) - 1, 1)) if ni > 1 else np.nan
         for ni in n],
        index=mean.index,
    )
    margin = t_mult * std / np.sqrt(n.replace(0, np.nan))
    return mean, mean - margin, mean + margin

# ── Colour map: curated palette + stable fallback cycle ──────────────────
_HARDCODED = {
    'no-adv-training': '#E8743B', 'adv-training': '#19A979',
    'FL_no-adv-training': '#2C7FB8', 'FL_adv-training': '#7FBC41',
    'no-adv / no-FL': '#E8743B', 'adv / no-FL': '#19A979',
    'no-adv / FL': '#2C7FB8', 'adv / FL': '#7FBC41',
    'FedAvg': '#3F51B5', 'FedProx': '#009688', 'FedYogi': '#E91E63',
    'mlp': '#2196F3', 'cnn': '#FF9800', 'resnet': '#4CAF50',
    'angela': '#2E86AB', 'bob': '#19A979', 'claude': '#E8743B', 'daniel': '#A23B72',
    'FL (dynamic-noise-fl)': '#2C7FB8', 'no-FL (dynamic-noise-nofl)': '#E8743B',
}
_fallback = itertools.cycle(plt.colormaps.get_cmap('tab10').colors)
_color_cache = {}
def color_for(label):
    if label in _HARDCODED:
        return _HARDCODED[label]
    if label not in _color_cache:
        _color_cache[label] = next(_fallback)
    return _color_cache[label]

def _style_ax(ax, title, xlabel, ylabel=None, ylim=None):
    ax.set_title(title, **title_font)
    ax.set_xlabel(xlabel, **axis_font)
    if ylabel:
        ax.set_ylabel(ylabel, **axis_font)
    ax.tick_params(axis='both', labelsize=axis_font['size'])
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.4)

def _plot_grid(panels, n_cols=2, suptitle='', figsize_per_cell=(9.0, 4.3),
               ci_alpha=0.95, show_ci_legend=True):
    # panels: list of {title, xlabel, ylabel, ylim, series:{label: wide_df}}
    panels = [p for p in panels if any(v is not None and not v.empty
                                       for v in p['series'].values())]
    if not panels:
        print('  (no data to plot for:', suptitle, ')')
        return
    n = len(panels)
    n_rows = ceil(n / n_cols)
    fw, fh = figsize_per_cell
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(fw * n_cols, fh * n_rows), squeeze=False)
    axes_flat = axes.flatten()
    ordered_labels = []
    for i, panel in enumerate(panels):
        ax = axes_flat[i]
        for label, wide in panel['series'].items():
            mean, lo, hi = compute_ci_band(wide, ci_alpha)
            if mean is None:
                continue
            c = color_for(label)
            ax.plot(mean.index, mean.values, label=label, color=c, lw=2.4, alpha=0.95)
            if lo is not None:
                ax.fill_between(lo.index, lo.values, hi.values, color=c, alpha=0.18)
            if label not in ordered_labels:
                ordered_labels.append(label)
        _style_ax(ax, panel.get('title', ''), panel.get('xlabel', 'Reporting step'),
                  panel.get('ylabel'), panel.get('ylim'))
    for j in range(n, len(axes_flat)):
        axes_flat[j].set_visible(False)

    handles = [Line2D([0], [0], color=color_for(l), lw=3.2, label=l) for l in ordered_labels]
    if show_ci_legend:
        handles.append(mpatches.Patch(alpha=0.3, color='grey',
                       label=f'mean ± {int(ci_alpha*100)} % CI (across seeds)'))
    fig.legend(handles=handles, prop=legend_font, loc='lower center',
               bbox_to_anchor=(0.5, -0.06), ncol=min(len(handles), 4), frameon=True)
    if suptitle:
        fig.suptitle(suptitle, weight='bold', size=24, y=1.015)
    plt.tight_layout()
    plt.show()

print('Plotting engine ready')

### 5.1 — High-level wrappers

- `compare_groups_per_vehicle` — one panel **per vehicle**, overlaying several
  experiment groups for a single metric (the Fig. 4 / 6 / 7 / 8 pattern, generalised).
- `metrics_per_vehicle_for_group` — for one experiment, one panel **per metric**,
  overlaying the four vehicles.
- `final_value_bars` — bar chart of the converged (last-`tail`-epochs) value per
  vehicle, grouped by configuration, with 95 % CI error bars.

Plus small helpers building the comparison group-maps.

In [ ]:
def compare_groups_per_vehicle(group_map, suffix, suptitle,
                               vehicles=VEHICLES, smooth=2, ylim=None,
                               ylabel=None, xlabel='Adv. benchmark round / epoch'):
    panels = []
    for v in vehicles:
        key = metric_key(v, suffix)
        series = {lab: seed_series_for_group(g, key, smooth) for lab, g in group_map.items()}
        panels.append({
            'title': f'{v.capitalize()}  (noise sigma={VEHICLE_NOISE[v]})',
            'series': series,
            'ylim': ylim if ylim is not None else ylim_for(suffix),
            'ylabel': ylabel or pretty(suffix),
            'xlabel': xlabel,
        })
    _plot_grid(panels, n_cols=2, suptitle=suptitle)

def metrics_per_vehicle_for_group(group, suffixes, suptitle,
                                  vehicles=VEHICLES, smooth=2,
                                  xlabel='Reporting step (per vehicle)'):
    if group not in GROUPS_PRESENT:
        print(f'  (group "{group}" not present — skipping "{suptitle}")')
        return
    panels = []
    for suf in suffixes:
        series = {v: seed_series_for_group(group, metric_key(v, suf), smooth) for v in vehicles}
        panels.append({
            'title': pretty(suf),
            'series': series,
            'ylim': ylim_for(suf),
            'ylabel': pretty(suf),
            'xlabel': xlabel,
        })
    _plot_grid(panels, n_cols=2, suptitle=suptitle)

def final_value_bars(group_map, suffix, suptitle, vehicles=VEHICLES,
                     smooth=3, tail=3, ylim=None):
    labels = list(group_map)
    means = np.full((len(labels), len(vehicles)), np.nan)
    errs  = np.full((len(labels), len(vehicles)), np.nan)
    any_data = False
    for li, lab in enumerate(labels):
        g = group_map[lab]
        for vi, v in enumerate(vehicles):
            wide = seed_series_for_group(g, metric_key(v, suffix), smooth)
            if wide is None:
                continue
            finals = wide.tail(tail).mean(axis=0).dropna().values   # one converged value per seed
            if len(finals) == 0:
                continue
            any_data = True
            means[li, vi] = np.mean(finals)
            if len(finals) > 1:
                se = np.std(finals, ddof=1) / np.sqrt(len(finals))
                errs[li, vi] = scipy_stats.t.ppf(0.975, df=len(finals) - 1) * se
    if not any_data:
        print('  (no data for bar chart:', suptitle, ')')
        return
    fig, ax = plt.subplots(figsize=(max(9, 2.6 * len(vehicles)), 6))
    x = np.arange(len(vehicles))
    w = 0.8 / max(len(labels), 1)
    for li, lab in enumerate(labels):
        ax.bar(x + li * w - 0.4 + w / 2, means[li], w, yerr=errs[li], capsize=5,
               label=lab, color=color_for(lab), alpha=0.92,
               edgecolor='white', linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{v}\nsigma={VEHICLE_NOISE[v]}' for v in vehicles],
                       fontsize=14, fontweight='bold')
    _style_ax(ax, suptitle, 'Vehicle', pretty(suffix), ylim if ylim is not None else ylim_for(suffix))
    ax.legend(prop=legend_font)
    plt.tight_layout()
    plt.show()

# ── Comparison group-maps ────────────────────────────────────────────────
ADV_CONTRAST_NOFL = {'no-adv-training': 'noadvtraining-nofl', 'adv-training': 'advtraining-nofl'}
ADV_CONTRAST_FL   = {'FL_no-adv-training': 'noadvtraining-fl', 'FL_adv-training': 'advtraining-fl'}
FOUR_CONFIGS      = {'no-adv / no-FL': 'noadvtraining-nofl', 'adv / no-FL': 'advtraining-nofl',
                     'no-adv / FL': 'noadvtraining-fl', 'adv / FL': 'advtraining-fl'}
FED_NOADV = {'FedAvg': 'noadvtraining-fl', 'FedProx': 'noadvtraining-fedprox', 'FedYogi': 'noadvtraining-fedyogi'}
FED_ADV   = {'FedAvg': 'advtraining-fl', 'FedProx': 'advtraining-fedprox', 'FedYogi': 'advtraining-fedyogi'}
DYNAMIC   = {'FL (dynamic-noise-fl)': 'dynamic-noise-fl', 'no-FL (dynamic-noise-nofl)': 'dynamic-noise-nofl'}

def model_groups(base):
    # {model: group_name} for one base experiment across architectures
    return {m: (base if m == 'mlp' else f'{base}-{m}') for m in MODELS}

def with_model(group_map, model):
    # Re-target a group-map onto a given architecture (append -cnn / -resnet)
    suf = '' if model == 'mlp' else f'-{model}'
    return {k: (v + suf) for k, v in group_map.items()}

print('Wrappers ready')

## 6 — ET1 / ET2: Adversarial robustness (Gaussian noise), **no FL**

Experiments #1 (`noadvtraining-nofl`) vs #2 (`advtraining-nofl`). One panel per
vehicle, overlaying adv-training on/off. This is the generalisation of the
paper's Fig. 4 (Accuracy) to **all** Gaussian adv-eval metrics.

In [ ]:
for suf in ADV_GAUSS_METRICS:
    compare_groups_per_vehicle(
        ADV_CONTRAST_NOFL, suf,
        suptitle=f'Gaussian adv-eval {pretty(suf)} — no FL (ET1/ET2)',
        smooth=2)

### 6.1 — ET1 / ET2 under the **HSJA** decision-based attack (no FL)

Worst-case black-box robustness. `avg_perturbation` (mean L2 distance to the
decision boundary) and `avg_queries` are the key robustness signals: higher
perturbation = boundary further from clean data = more robust.

In [ ]:
for suf in HSJA_METRICS:
    compare_groups_per_vehicle(
        ADV_CONTRAST_NOFL, suf,
        suptitle=f'HSJA {pretty(suf)} — no FL (ET1/ET2)',
        smooth=2)

## 7 — ET3: Federated robustness transfer

Experiments #3 (`noadvtraining-fl`) vs #6 (`advtraining-fl`). Generalises the
paper's Fig. 6: does robustness learned by Bob/Claude/Daniel propagate to Angela
(trained clean) through FedAvg aggregation? Shown for every Gaussian and HSJA
metric.

In [ ]:
for suf in ADV_GAUSS_METRICS:
    compare_groups_per_vehicle(
        ADV_CONTRAST_FL, suf,
        suptitle=f'Gaussian adv-eval {pretty(suf)} — FedAvg FL (ET3)',
        smooth=2)

for suf in HSJA_METRICS:
    compare_groups_per_vehicle(
        ADV_CONTRAST_FL, suf,
        suptitle=f'HSJA {pretty(suf)} — FedAvg FL (ET3)',
        smooth=2)

## 8 — ET4: In-distribution online-monitoring performance

Experiments #1/#2 (no FL) and #3/#6 (FL), evaluated in the online-monitoring
loop. Generalises the paper's Fig. 7 & Fig. 8 to all online metrics — verifying
that robustness mechanisms do not tax nominal accuracy. Online metrics are dense,
so a larger smoothing window is used.

In [ ]:
for suf in ONLINE_METRICS:
    compare_groups_per_vehicle(
        ADV_CONTRAST_NOFL, suf,
        suptitle=f'Online monitoring {pretty(suf)} — no FL (ET4)',
        smooth=6, xlabel='Reporting epoch')

for suf in ONLINE_METRICS:
    compare_groups_per_vehicle(
        ADV_CONTRAST_FL, suf,
        suptitle=f'Online monitoring {pretty(suf)} — FedAvg FL (ET4)',
        smooth=6, xlabel='Reporting epoch')

In [ ]:
for suf in ONLINE_METRICS:
    compare_groups_per_vehicle(
        FED_NOADV, suf,
        suptitle=f'Online monitoring {pretty(suf)} — FL strategy (no adv-training)',
        smooth=6, xlabel='Reporting epoch')

for suf in ONLINE_METRICS:
    compare_groups_per_vehicle(
        FED_ADV, suf,
        suptitle=f'Online monitoring {pretty(suf)} — FL strategy (adv-training)',
        smooth=6, xlabel='Reporting epoch')

## 9 — The four canonical configurations together

Overlays all of #1/#2/#3/#6 on a single panel per vehicle, for the headline
metrics — the most compact view of the adv-training × FL interaction.

In [ ]:
for suf in ['adv_eval_accuracy', 'adv_eval_macro_f1',
            'hsja_adv_eval/accuracy', 'hsja_adv_eval/avg_perturbation',
            'online_class_accuracy']:
    compare_groups_per_vehicle(
        FOUR_CONFIGS, suf,
        suptitle=f'Four configurations — {pretty(suf)}',
        smooth=3)

## 10 — FL aggregation strategy: FedAvg vs FedProx vs FedYogi

Experiments #3/#4/#5 (no adv-training)
and #6/#7/#8 (adv-training) isolate the effect of the aggregation rule at a fixed training regime.

In [ ]:
for suf in ['adv_eval_accuracy', 'adv_eval_macro_f1',
            'hsja_adv_eval/accuracy', 'hsja_adv_eval/avg_perturbation',
            'online_class_accuracy']:
    compare_groups_per_vehicle(
        FED_NOADV, suf,
        suptitle=f'FL strategy (no adv-training) — {pretty(suf)}',
        smooth=3)

for suf in ['adv_eval_accuracy', 'adv_eval_macro_f1',
            'hsja_adv_eval/accuracy', 'hsja_adv_eval/avg_perturbation',
            'online_class_accuracy']:
    compare_groups_per_vehicle(
        FED_ADV, suf,
        suptitle=f'FL strategy (adv-training) — {pretty(suf)}',
        smooth=3)

## 11 — Classifier architecture: MLP vs CNN vs ResNet

Beyond the paper (which uses only an MLP). For several base experiments, overlay
the three architectures per vehicle — answering review comment RC 2.3. Because
all three share the same 2D manifold + linear output, HSJA metrics are directly
comparable across architectures.

In [ ]:
ARCH_BASES = ['noadvtraining-nofl', 'advtraining-nofl', 'noadvtraining-fl', 'advtraining-fl']
ARCH_METRICS = ['adv_eval_accuracy', 'adv_eval_macro_f1',
                'hsja_adv_eval/accuracy', 'hsja_adv_eval/avg_perturbation',
                'online_class_accuracy']

for base in ARCH_BASES:
    for suf in ARCH_METRICS:
        compare_groups_per_vehicle(
            model_groups(base), suf,
            suptitle=f'Architecture comparison — {base} — {pretty(suf)}',
            smooth=3)

## 12 — Per-experiment metric panoramas (all metrics, all vehicles)

For a few headline experiments, draw one panel per metric with the four vehicles
overlaid — a single-figure overview of an experiment's full metric family.
Repeated for Gaussian adv-eval, HSJA, online monitoring and training metrics.

In [ ]:
PANORAMA_GROUPS = ['noadvtraining-nofl', 'advtraining-nofl', 'advtraining-fl']

for grp in PANORAMA_GROUPS:
    metrics_per_vehicle_for_group(grp, ADV_GAUSS_METRICS,
        suptitle=f'{grp} — Gaussian adv-eval metrics (per vehicle)', smooth=2)
    metrics_per_vehicle_for_group(grp, HSJA_METRICS,
        suptitle=f'{grp} — HSJA metrics (per vehicle)', smooth=2)
    metrics_per_vehicle_for_group(grp, ONLINE_METRICS,
        suptitle=f'{grp} — online-monitoring metrics (per vehicle)', smooth=6)
    metrics_per_vehicle_for_group(grp, TRAINING_METRICS,
        suptitle=f'{grp} — training metrics (per vehicle)', smooth=4)

## 13 — Attack-mitigation operational metrics

`mitigation_reward` (TP/FP/TN/FN-weighted operational quality) and
`mitigation_time` (end-to-end response latency), per vehicle, across the four
canonical configurations.

In [ ]:
for suf in MITIGATION_METRICS:
    compare_groups_per_vehicle(
        FOUR_CONFIGS, suf,
        suptitle=f'{pretty(suf)} — four configurations',
        smooth=4, xlabel='Reporting epoch')

## 14 — Dynamic-noise runs (Angela: noise 0 → 1 mid-run)

Experiments #9 (`dynamic-noise-fl`) vs #10 (`dynamic-noise-nofl`). Angela's noise
is injected at ~50 % of the run. The discontinuity should be sharper without FL;
with FL, aggregation cushions the drop. Focused on Angela, then all vehicles for
context.

In [ ]:
DN_METRICS = ['hsja_adv_eval/avg_perturbation', 'hsja_adv_eval/accuracy',
              'adv_eval_accuracy', 'adv_eval_macro_f1', 'online_class_accuracy']

for suf in DN_METRICS:
    compare_groups_per_vehicle(
        DYNAMIC, suf, vehicles=['angela'],
        suptitle=f'Dynamic noise on Angela — {pretty(suf)}  (FL vs no-FL)',
        smooth=2, xlabel='Reporting step (0 -> 50% : noise 0, then noise 1.0)')

# Full-fleet context for the headline robustness signal
compare_groups_per_vehicle(
    DYNAMIC, 'hsja_adv_eval/avg_perturbation',
    suptitle='Dynamic noise — HSJA avg L2 perturbation (all vehicles)',
    smooth=2)

## 15 — Converged-value summary bar charts

Bar charts of the converged (last-3-reporting-epochs) value per vehicle, with
95 % CI error bars across seeds — compact "final scoreboard" views.

In [ ]:
final_value_bars(ADV_CONTRAST_NOFL, 'adv_eval_accuracy',
                 'Converged Gaussian adv-eval accuracy — no FL')
final_value_bars(ADV_CONTRAST_FL, 'adv_eval_accuracy',
                 'Converged Gaussian adv-eval accuracy — FedAvg FL')
final_value_bars(FOUR_CONFIGS, 'hsja_adv_eval/accuracy',
                 'Converged HSJA accuracy — four configurations')
final_value_bars(FED_NOADV, 'hsja_adv_eval/avg_perturbation',
                 'Converged HSJA L2 perturbation by FL strategy (no adv-training)')
final_value_bars(model_groups('noadvtraining-nofl'), 'hsja_adv_eval/avg_perturbation',
                 'Converged HSJA L2 perturbation by architecture (baseline)')
final_value_bars(model_groups('advtraining-nofl'), 'adv_eval_macro_f1',
                 'Converged adv-eval Macro-F1 by architecture (adv-training)')

## 16 — Qualitative artifacts (Plotly media) — index only

The manifold / confusion-matrix figures (`*_manifold_pca`, `*_manifold_labels`,
`*_manifold_preds`, `*_adv_eval_confusion_matrix`, `*_hsja_*`,
`*_online_confusion_matrix`) are logged as W&B **media**, not scalar history, so
they are viewed in the W&B UI rather than re-rendered here. The cell below lists,
per group/seed, which media keys exist and prints a direct W&B URL.

In [ ]:
def list_media_artifacts(max_runs=None):
    api = wandb.Api(timeout=60)
    runs = api.runs(WANDB_PROJECT)
    shown = 0
    for run in runs:
        if GROUPS_FILTER and run.group not in GROUPS_FILTER:
            continue
        media_keys = []
        for k, v in dict(run.summary).items():
            if any(tag in k for tag in ('manifold', 'confusion_matrix')):
                media_keys.append(k)
        if media_keys:
            print(f'\n{run.group} / {run.name}\n  {run.url}')
            for k in sorted(media_keys):
                print('   ', k)
            shown += 1
            if max_runs and shown >= max_runs:
                break
    if shown == 0:
        print('No media artifacts found (or runs not yet synced).')

# Uncomment to enumerate (makes extra W&B API calls):
# list_media_artifacts(max_runs=10)
print('Run list_media_artifacts() to index manifold / confusion-matrix media.')

In [ ]:
list_media_artifacts()

## 17 — Render the actual Plotly artifacts of a single run (restyled)

The manifold scatters and confusion matrices are logged to W&B as **Plotly
media**, not scalar history — so this section pulls the *raw* Plotly figures back
down for **one explicitly-chosen run + seed** and re-renders them locally, with
**bold ticks, enlarged fonts and configurable dimensions**, ready for you to
fine-tune before exporting.

For the selected run it downloads, **for every plot type and every vehicle, the
last figure that was logged** (W&B's run *summary* always holds the most recent
value of each key, i.e. the final benchmark round's plots). Each figure is shown
interactively (`fig.show()`) so you can tweak it, and a 2× PNG is written next to
it if `kaleido` is installed.

Plot types pulled per vehicle: `*_manifold_pca`, `*_manifold_labels`,
`*_manifold_preds`, `*_adv_eval_confusion_matrix`, `*_online_confusion_matrix`,
`*_hsja_manifold_pca`, `*_hsja_manifold_labels`, `*_hsja_manifold_preds`,
`*_hsja_adv_eval_confusion_matrix`.

> Requires `plotly` (and optionally `kaleido` for static PNG export):
> `pip install plotly kaleido`. Tweak the knobs in the next cell, then pick the
> run/seed/vehicles/plots you actually care about.

In [ ]:
# pip install plotly kaleido    # if not already available
import tempfile
import json as _json
import base64
from pathlib import Path

import plotly
import plotly.io as pio
import plotly.graph_objects as go

# ── 1. Choose ONE run ────────────────────────────────────────────────────
PLOT_GROUP    = 'advtraining-nofl'     # any W&B group (see "Groups present" above)
PLOT_SEED     = 42                     # one of DEFAULT_SEEDS
PLOT_VEHICLES = VEHICLES               # e.g. ['daniel'] to focus on one vehicle

# Limit which plot types to pull (None = all nine). E.g. ['manifold_preds'].
PLOT_TYPES = None

# ── 2. Styling knobs (this is what you came here to tune) ────────────────
PLOT_W, PLOT_H   = 900, 700            # figure pixel dimensions
TICK_SIZE        = 18
AXIS_TITLE_SIZE  = 20
TITLE_SIZE       = 22
LEGEND_SIZE      = 16
MARKER_SIZE      = 15
FONT_FAMILY      = 'DejaVu Sans, Arial, sans-serif'
SAVE_PNG         = True                # write a 2x static PNG next to each figure

# Plot-type suffixes (key = f"{vehicle}_{suffix}")
PLOTLY_SUFFIXES = [
    'manifold_pca', 'manifold_labels', 'manifold_preds',
    'adv_eval_confusion_matrix', 'online_confusion_matrix',
    'hsja_manifold_pca', 'hsja_manifold_labels', 'hsja_manifold_preds',
    'hsja_adv_eval_confusion_matrix',
]

# Plotly gained per-element font `weight` in 5.22; gate on it for bold ticks.
_PLOTLY_WEIGHT = tuple(int(x) for x in plotly.__version__.split('.')[:2]) >= (5, 22)
def _bf(size, bold=True):
    d = {'size': size, 'family': FONT_FAMILY}
    if bold and _PLOTLY_WEIGHT:
        d['weight'] = 'bold'
    return d

# ── 3. Locate the run object via the W&B public API ──────────────────────
def find_run(group, seed):
    api = wandb.Api(timeout=60)
    runs = list(api.runs(WANDB_PROJECT, filters={'group': group}))
    cands = [r for r in runs if _seed_of(r) == seed]
    if WANDB_TAG_FILTER:
        tagged = [r for r in cands if any(t in r.tags for t in WANDB_TAG_FILTER)]
        cands = tagged or cands
    if not cands:
        return None
    # Prefer the most complete run (highest final step).
    cands.sort(key=lambda r: (r.summary.get('_step', 0) or 0), reverse=True)
    return cands[0]

# ── 4. Download + parse the last Plotly figure for a given key ───────────
def _decode_bdata(obj):
    # Plotly serialises numpy arrays as {'dtype':'f4','bdata':'<base64>','shape':'3, 3'}.
    # Rebuild them into real ndarrays so go.Figure() accepts them.
    if isinstance(obj, dict):
        if 'bdata' in obj and 'dtype' in obj:
            arr = np.frombuffer(base64.b64decode(obj['bdata']), dtype=np.dtype(obj['dtype']))
            shape = obj.get('shape')
            if shape:
                arr = arr.reshape([int(s) for s in str(shape).split(',')])
            return arr
        return {k: _decode_bdata(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_decode_bdata(v) for v in obj]
    return obj

def _load_plotly_file(path):
    raw = _json.loads(Path(path).read_text())
    d = _decode_bdata(raw)
    return go.Figure(data=d.get('data'), layout=d.get('layout'))

_PLOTLY_FILE_CACHE = {}   # run.id -> [File, ...]  (avoid re-listing per key)

def _plotly_files(run):
    if run.id not in _PLOTLY_FILE_CACHE:
        _PLOTLY_FILE_CACHE[run.id] = [
            f for f in run.files()
            if f.name.startswith('media/plotly/') and f.name.endswith('.json')
        ]
    return _PLOTLY_FILE_CACHE[run.id]

def download_last_plotly(run, key, dest):
    # Plotly media is stored as run files: media/plotly/{key}_{step}_{hash}.plotly.json
    prefix = f'media/plotly/{key}_'
    matches = [f for f in _plotly_files(run) if f.name.startswith(prefix)]
    if not matches:
        return None                      # key never logged for this run
    def _step(f):                        # pick the LAST logged (highest step)
        tail = f.name[len(prefix):]
        try:
            return int(tail.split('_')[0])
        except Exception:
            return -1
    f = max(matches, key=_step)
    f.download(root=str(dest), replace=True)
    return _load_plotly_file(Path(dest) / f.name)

# ── 5. Restyle: bold ticks, big fonts, fixed dimensions ──────────────────
def restyle(fig, title):
    fig.update_layout(
        width=PLOT_W, height=PLOT_H,
        template='plotly_white',
        title=dict(text=title, x=0.5, xanchor='center', font=_bf(TITLE_SIZE)),
        font=dict(size=TICK_SIZE, family=FONT_FAMILY),
        legend=dict(font=_bf(LEGEND_SIZE), borderwidth=1),
        margin=dict(l=80, r=40, t=64, b=70),
    )
    axis_kw = dict(
        tickfont=_bf(TICK_SIZE),
        title_font=_bf(AXIS_TITLE_SIZE),
        showline=True, linewidth=1.8, linecolor='black',
        ticks='outside', tickwidth=1.8, ticklen=6,
        showgrid=True, gridcolor='rgba(0,0,0,0.12)',
    )
    fig.update_xaxes(**axis_kw)
    fig.update_yaxes(**axis_kw)
    # Scatter markers a touch larger; confusion-matrix colorbar ticks bolder.
    fig.update_traces(marker=dict(size=MARKER_SIZE),
                      selector=dict(type='scatter'))
    try:
        fig.update_coloraxes(colorbar=dict(tickfont=_bf(TICK_SIZE)))
    except Exception:
        pass
    return fig

# ── 6. Drive it ──────────────────────────────────────────────────────────
suffixes = [s for s in PLOTLY_SUFFIXES if (PLOT_TYPES is None or s in PLOT_TYPES)]
run = find_run(PLOT_GROUP, PLOT_SEED)

if run is None:
    print(f'No run found for group="{PLOT_GROUP}", seed={PLOT_SEED}.')
    avail = sorted(RUNS.keys())
    print('Available (group, seed) in the current fetch:')
    for g, s in avail:
        print(f'   {g:30s} seed={s}')
else:
    dest = Path(tempfile.mkdtemp(prefix='serebench_plotly_'))
    print(f'Selected run : {run.name}')
    print(f'   group={run.group}  seed={PLOT_SEED}  url={run.url}')
    print(f'   downloading last figures into {dest}\n')
    n_ok, n_missing = 0, 0
    for v in PLOT_VEHICLES:
        for suf in suffixes:
            key = f'{v}_{suf}'
            try:
                fig = download_last_plotly(run, key, dest)
            except Exception as exc:
                print(f'  ! {key}: download/parse failed ({exc})')
                n_missing += 1
                continue
            if fig is None:
                n_missing += 1
                continue
            restyle(fig, key)
            print(f'  + {key}')
            fig.show()
            if SAVE_PNG:
                try:
                    fig.write_image(str(dest / f'{key}.png'), scale=2)
                except Exception as exc:
                    print(f'    (PNG export skipped — install kaleido: {exc})')
            n_ok += 1
    print(f'\nRendered {n_ok} figures, {n_missing} keys absent for this run.')
    print(f'PNGs (if any) saved under: {dest}')